# tACS Bandit EEG Analyses (v5)

Tier 2 EEG analysis notebook for: *Examining the effects of theta-tACS over left DLPFC on reward-based learning across the adult lifespan*

**v5 Updates:**
- Multi-channel ITF analysis: F4, P4, P3 all analyzed with equal priority
- Fixed specparam plotting (model components now render correctly)
- Cross-participant theta detection summary
- Descriptive filenames for all saved plots
- Better handling of edge cases in PSD computation

**Previous fixes retained:**
- v4: Absolute threshold stimulation detection (working correctly)
- v3: Artifact rejection on filtered data
- v3: Reference-aware preprocessing

**Known Data Issues:**
- sub-10998: Missing Run 1; Run 6 was sham (experimenter error)
- sub-11773: Runs 6-7 were sham (experimenter error)
- sub-10951: Electrode artifact (loose connection)

## 0. Setup and Configuration

In [ ]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal, stats
from scipy.signal import butter, filtfilt, iirnotch
from pathlib import Path
from specparam import SpectralModel
from datetime import datetime

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='loadtxt: input contained no data')

# --- Path configuration ---
REPO_ROOT = Path('..').resolve()
EEG_DIR = REPO_ROOT / 'data' / 'nic' / 'raw'
OUTPUT_DIR = REPO_ROOT / 'derivatives' / 'eeg'
PLOT_DIR = OUTPUT_DIR / 'plots'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# --- Subject tracker ---
SUBJECT_INFO = {
    '10886': {'counterbalance': 'B', 'earclip': False, 'notes': ''},
    '10998': {'counterbalance': 'B', 'earclip': False, 'notes': 'Missing Run 1; Run 6 was sham (experimenter error)'},
    '11773': {'counterbalance': 'B', 'earclip': False, 'notes': 'Runs 6-7 were sham (experimenter error)'},
    '10656': {'counterbalance': 'A', 'earclip': False, 'notes': ''},
    '10951': {'counterbalance': 'A', 'earclip': False, 'notes': 'Electrode artifact (loose connection suspected)'},
    '10418': {'counterbalance': 'B', 'earclip': False, 'notes': ''},
    '10636': {'counterbalance': 'B', 'earclip': True, 'notes': ''},
    '11318': {'counterbalance': 'B', 'earclip': True, 'notes': ''},
    '10369': {'counterbalance': 'B', 'earclip': True, 'notes': ''},
    '10606': {'counterbalance': 'A', 'earclip': True, 'notes': ''},
    '11628': {'counterbalance': 'A', 'earclip': True, 'notes': ''},
    '11286': {'counterbalance': 'A', 'earclip': True, 'notes': ''},
    '10866': {'counterbalance': 'B', 'earclip': True, 'notes': ''},
    '11329': {'counterbalance': 'B', 'earclip': True, 'notes': ''}
}

# --- Run structure ---
RUN_CONDITIONS = {
    'A': {1: 'baseline', 2: 'active', 3: 'active', 4: 'post',
          5: 'baseline', 6: 'sham', 7: 'sham', 8: 'post'},
    'B': {1: 'baseline', 2: 'sham', 3: 'sham', 4: 'post',
          5: 'baseline', 6: 'active', 7: 'active', 8: 'post'},
}

# --- EEG constants ---
FS = 500
CH_LABELS = ['F3', 'Fp1', 'FCz', 'FT7', 'F4', 'P4', 'P3', 'Ch8']
STIM_CH_IDX = [0, 1, 2, 3]
EEG_CH_IDX = [4, 5, 6]
EEG_CH_LABELS = ['F4', 'P4', 'P3']
TACS_FREQ = 6.0

# --- Frequency bands ---
BANDS = {
    'delta': (1, 4),
    'theta': (4, 8),
    'alpha': (8, 13),
    'beta':  (13, 30),
    'gamma': (30, 50),
}

# --- Preprocessing parameters ---
PREPROC_PARAMS = {
    'earclip': {
        'apply_avg_reref': False,
        'highpass_cutoff': 0.5,
        'lowpass_cutoff': 40.0,
        'notch_freq': 60.0,
        'artifact_threshold_uv': 150,
    },
    'no_earclip': {
        'apply_avg_reref': True,
        'highpass_cutoff': 0.5,
        'lowpass_cutoff': 40.0,
        'notch_freq': 60.0,
        'artifact_threshold_uv': 150,
    }
}

# --- Stimulation detection thresholds ---
STIM_THRESHOLDS = {
    'earclip': {
        'active_middle_min_db': 90,
        'active_sustained_diff_db': 5,
        'sham_early_min_db': 85,
        'sham_early_late_diff_db': 10,
    },
    'no_earclip': {
        'active_middle_min_db': 120,
        'active_sustained_diff_db': 10,
        'sham_early_min_db': 100,
        'sham_early_late_diff_db': 15,
    }
}

# --- ITF extraction parameters ---
ITF_PARAMS = {
    'r_squared_threshold': 0.70,
    'theta_range': (4, 8),
    'alpha_range': (8, 13),  # Also track alpha for reference
    'specparam_settings': {
        'peak_width_limits': (0.5, 6),
        'max_n_peaks': 6,
        'min_peak_height': 0.05,  # Even lower to catch subtle peaks
        'aperiodic_mode': 'fixed',
    }
}

print(f'Repository root: {REPO_ROOT}')
print(f'EEG directory: {EEG_DIR}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'Plot directory: {PLOT_DIR}')
print(f'\nSubjects: {len(SUBJECT_INFO)}')
print(f'  Earclip: {[s for s, info in SUBJECT_INFO.items() if info["earclip"]]}')
print(f'  No-earclip: {[s for s, info in SUBJECT_INFO.items() if not info["earclip"]]}')

issues = [(s, info['notes']) for s, info in SUBJECT_INFO.items() if info['notes']]
if issues:
    print(f'\n⚠ Known issues:')
    for s, note in issues:
        print(f'  {s}: {note}')

## 1. Utility Functions

In [ ]:
# =============================================================================
# 1a. Plot Saving Utility
# =============================================================================

def save_plot(fig, name, subject_id=None, format='png'):
    """
    Save a plotly figure with a descriptive filename.
    
    Filename format: {name}_sub-{subject_id}_{date}.{format}
    or: {name}_{date}.{format} if no subject_id
    """
    date_str = datetime.now().strftime('%Y%m%d')
    
    if subject_id:
        filename = f'{name}_sub-{subject_id}_{date_str}.{format}'
    else:
        filename = f'{name}_{date_str}.{format}'
    
    filepath = PLOT_DIR / filename
    
    if format == 'png':
        fig.write_image(str(filepath), scale=2)
    elif format == 'html':
        fig.write_html(str(filepath))
    else:
        fig.write_image(str(filepath))
    
    print(f'Saved: {filepath}')
    return filepath


# =============================================================================
# 1b. Data Loading
# =============================================================================

def load_nic2_run(easy_path, info_path=None, imp_path=None):
    """Load a single NIC2 recording."""
    raw = np.loadtxt(easy_path)
    eeg = raw[:, :8]
    triggers = raw[:, 11] if raw.shape[1] > 11 else np.zeros(len(raw))
    timestamps = raw[:, 12] if raw.shape[1] > 12 else np.arange(len(raw))
    
    info = {}
    if info_path and Path(info_path).exists():
        with open(info_path, 'r') as f:
            info_text = f.read()
        for line in info_text.split('\n'):
            if 'EEG sampling rate' in line and 'Samples' in line:
                info['fs'] = int(line.split(':')[1].strip().split()[0])
            if 'Step name' in line:
                info['step_name'] = line.split(':')[1].strip()
            if 'Atacs (uA)' in line:
                val = int(line.split(':')[1].strip())
                if val > 0:
                    info['has_active_stim'] = True
    
    fs = info.get('fs', FS)
    stim_mask = (eeg[:, 0] == -1)
    
    impedance = None
    if imp_path and Path(imp_path).exists():
        try:
            imp_raw = np.loadtxt(imp_path)
            if imp_raw.ndim == 2 and imp_raw.shape[0] > 0:
                impedance = {
                    'values': imp_raw[:, :8],
                    'mean_per_channel': np.mean(imp_raw[:, :8], axis=0),
                }
        except Exception:
            pass
    
    return {
        'eeg': eeg, 'timestamps': timestamps, 'triggers': triggers,
        'fs': fs, 'stim_mask': stim_mask, 'impedance': impedance, 'info': info,
    }


def discover_runs(eeg_dir, subject_id):
    """Find all NIC2 files for a subject."""
    pattern = str(eeg_dir / f'*sub-{subject_id}_Run*')
    all_files = glob.glob(pattern + '.easy')
    
    runs = {}
    for easy_path in sorted(all_files):
        easy_path = Path(easy_path)
        fname = easy_path.stem
        
        try:
            run_str = fname.split('Run')[1].strip().split('_')[0].strip()
            run_num = int(run_str)
        except (IndexError, ValueError):
            continue
        
        info_path = easy_path.with_suffix('.info')
        imp_candidates = list(eeg_dir.glob(f'*sub-{subject_id}_Run*{run_num}_IMP.txt'))
        imp_path = imp_candidates[0] if imp_candidates else None
        
        file_size = easy_path.stat().st_size
        
        if run_num not in runs or file_size > runs[run_num]['size']:
            runs[run_num] = {
                'easy': easy_path,
                'info': info_path if info_path.exists() else None,
                'imp': imp_path,
                'size': file_size,
            }
    
    for run in runs.values():
        del run['size']
    
    return runs


print('Utility functions ready.')

## 2. Preprocessing

In [ ]:
def apply_filters(eeg_data, fs=FS, highpass=0.5, lowpass=40.0, notch=60.0):
    """Apply bandpass and notch filters."""
    nyq = fs / 2
    b, a = butter(4, [highpass / nyq, lowpass / nyq], btype='band')
    
    if eeg_data.ndim == 1:
        filtered = filtfilt(b, a, eeg_data.astype(float))
    else:
        filtered = np.zeros_like(eeg_data, dtype=float)
        for ch in range(eeg_data.shape[1]):
            filtered[:, ch] = filtfilt(b, a, eeg_data[:, ch].astype(float))
    
    if notch is not None:
        b_notch, a_notch = iirnotch(notch, Q=30, fs=fs)
        if filtered.ndim == 1:
            filtered = filtfilt(b_notch, a_notch, filtered)
        else:
            for ch in range(filtered.shape[1]):
                filtered[:, ch] = filtfilt(b_notch, a_notch, filtered[:, ch])
    
    return filtered


def avg_rereference(eeg, ch_indices=EEG_CH_IDX):
    """Average re-reference across specified channels."""
    subset = eeg[:, ch_indices].astype(float)
    avg = np.mean(subset, axis=1, keepdims=True)
    return subset - avg


def reject_artifacts(eeg_data, fs=FS, threshold_uv=150, highpass=0.5, 
                     detect_pops=True, pop_threshold_uv=500):
    """Artifact rejection on filtered data."""
    eeg_uv = eeg_data / 1000
    
    nyq = fs / 2
    b, a = butter(2, highpass / nyq, btype='high')
    eeg_filtered = filtfilt(b, a, eeg_uv.astype(float))
    
    amp_clean = np.abs(eeg_filtered) < threshold_uv
    
    if detect_pops:
        diff = np.abs(np.diff(eeg_uv))
        pop_mask = np.concatenate([[False], diff > pop_threshold_uv])
        extend_samples = int(0.05 * fs)
        pop_indices = np.where(pop_mask)[0]
        for idx in pop_indices:
            start = max(0, idx - extend_samples)
            end = min(len(pop_mask), idx + extend_samples)
            pop_mask[start:end] = True
        pop_clean = ~pop_mask
    else:
        pop_clean = np.ones(len(eeg_uv), dtype=bool)
    
    clean_mask = amp_clean & pop_clean
    pct_rejected = 100 * (1 - clean_mask.mean())
    
    return clean_mask, pct_rejected, {
        'pct_amplitude': 100 * (1 - amp_clean.mean()),
        'pct_pops': 100 * (1 - pop_clean.mean()) if detect_pops else 0,
        'pct_total': pct_rejected,
    }


def preprocess_eeg(eeg_raw, has_earclip, fs=FS):
    """Full preprocessing pipeline."""
    params = PREPROC_PARAMS['earclip' if has_earclip else 'no_earclip']
    
    eeg_subset = eeg_raw[:, EEG_CH_IDX].astype(float)
    
    if params['apply_avg_reref']:
        eeg_reref = avg_rereference(eeg_raw)
    else:
        eeg_reref = eeg_subset
    
    eeg_filtered = apply_filters(
        eeg_reref, fs=fs,
        highpass=params['highpass_cutoff'],
        lowpass=params['lowpass_cutoff'],
        notch=params['notch_freq']
    )
    
    # Use F4 for artifact detection, apply mask to all channels
    clean_mask, pct_rejected, rejection_info = reject_artifacts(
        eeg_filtered[:, 0], fs=fs,
        threshold_uv=params['artifact_threshold_uv'],
        highpass=params['highpass_cutoff'],
    )
    
    return eeg_filtered, clean_mask, {
        'has_earclip': has_earclip,
        'params': params,
        'rejection': rejection_info,
        'n_samples_clean': np.sum(clean_mask),
        'n_samples_total': len(clean_mask),
    }


print('Preprocessing functions ready.')

## 3. Stimulation Detection

In [ ]:
def compute_6hz_timecourse(eeg_data, fs=FS):
    """Compute time-resolved 6 Hz power."""
    f, t_spec, Sxx = signal.spectrogram(
        signal.detrend(eeg_data), fs=fs,
        nperseg=1000, noverlap=750, scaling='density'
    )
    
    six_hz_mask = (f >= 5.5) & (f <= 6.5)
    power_db = 10 * np.log10(np.mean(Sxx[six_hz_mask, :], axis=0) + 1e-30)
    
    return t_spec, power_db


def compute_window_powers(eeg_data, fs=FS):
    """Compute 6 Hz power in early, middle, and late windows."""
    t_spec, power_db = compute_6hz_timecourse(eeg_data, fs)
    
    if len(t_spec) == 0:
        return {'error': 'No spectrogram'}
    
    duration = t_spec[-1]
    
    early_mask = t_spec <= 45
    middle_mask = (t_spec >= 60) & (t_spec <= min(300, duration - 45))
    late_mask = t_spec >= (duration - 45)
    
    return {
        'early_db': np.mean(power_db[early_mask]) if np.any(early_mask) else np.nan,
        'middle_db': np.mean(power_db[middle_mask]) if np.any(middle_mask) else np.nan,
        'late_db': np.mean(power_db[late_mask]) if np.any(late_mask) else np.nan,
        'duration_s': duration,
        't_spec': t_spec,
        'power_db': power_db,
    }


def classify_stimulation(eeg_data, has_earclip, fs=FS):
    """Classify run as ACTIVE, SHAM, or NONE."""
    thresholds = STIM_THRESHOLDS['earclip' if has_earclip else 'no_earclip']
    powers = compute_window_powers(eeg_data, fs)
    
    if 'error' in powers:
        return 'NONE', powers
    
    early = powers['early_db']
    middle = powers['middle_db']
    late = powers['late_db']
    
    middle_high = middle >= thresholds['active_middle_min_db']
    sustained = abs(middle - late) <= thresholds['active_sustained_diff_db']
    
    if middle_high and sustained:
        classification = 'ACTIVE'
    elif (early >= thresholds['sham_early_min_db'] and 
          (early - late) >= thresholds['sham_early_late_diff_db']):
        classification = 'SHAM'
    else:
        classification = 'NONE'
    
    powers['classification'] = classification
    powers['thresholds'] = thresholds
    
    return classification, powers


print('Stimulation detection ready.')

## 4. Spectral Parameterization (v5: Multi-Channel)

In [ ]:
def compute_psd(eeg_data, fs=FS, nperseg=2048):
    """
    Compute PSD using Welch's method.
    
    v5: Added better handling of edge cases.
    """
    # Ensure we have valid data
    if len(eeg_data) < 256:
        return None, None
    
    # Adjust nperseg if needed
    if len(eeg_data) < nperseg:
        nperseg = min(len(eeg_data), 1024)
        if nperseg < 256:
            nperseg = 256
    
    # Detrend and compute PSD
    try:
        data = signal.detrend(eeg_data.astype(float))
        freqs, psd = signal.welch(data, fs=fs, nperseg=nperseg, noverlap=nperseg//2)
        
        # Check for invalid values
        if np.any(np.isnan(psd)) or np.any(np.isinf(psd)):
            return None, None
        
        return freqs, psd
    except Exception as e:
        print(f'PSD computation error: {e}')
        return None, None


def fit_specparam(freqs, psd, freq_range=(1, 40)):
    """
    Fit specparam model.
    
    v5: Takes linear PSD (not log), handles conversion internally.
    """
    settings = ITF_PARAMS['specparam_settings']
    
    # Ensure PSD is positive before log transform
    psd_clean = np.maximum(psd, 1e-30)
    
    # Convert to log power for specparam
    # Note: specparam expects log10(power), not 10*log10(power)
    psd_log = np.log10(psd_clean)
    
    if np.any(np.isnan(psd_log)) or np.any(np.isinf(psd_log)):
        return None
    
    sm = SpectralModel(
        peak_width_limits=settings['peak_width_limits'],
        max_n_peaks=settings['max_n_peaks'],
        min_peak_height=settings['min_peak_height'],
        aperiodic_mode=settings['aperiodic_mode'],
        verbose=False,
    )
    
    try:
        sm.fit(freqs, psd_log, freq_range)
        return sm
    except Exception as e:
        return None


def extract_peaks(model, theta_range=(4, 8), alpha_range=(8, 13)):
    """
    Extract all peaks and identify theta/alpha peaks.
    
    v5: Returns more detailed peak information.
    """
    result = {
        'all_peaks': None,
        'theta_peak': None,
        'alpha_peak': None,
        'itf': None,
        'iaf': None,  # Individual Alpha Frequency
        'has_theta': False,
        'has_alpha': False,
    }
    
    if model is None:
        return result
    
    try:
        peaks = model.results.params.periodic.params
    except Exception:
        return result
    
    if peaks is None or (hasattr(peaks, '__len__') and len(peaks) == 0):
        return result
    
    # Ensure 2D
    if peaks.ndim == 1:
        peaks = peaks.reshape(1, -1)
    
    result['all_peaks'] = peaks
    
    # Find theta peaks
    theta_mask = (peaks[:, 0] >= theta_range[0]) & (peaks[:, 0] <= theta_range[1])
    if np.any(theta_mask):
        theta_peaks = peaks[theta_mask]
        best_theta = theta_peaks[np.argmax(theta_peaks[:, 1])]
        result['theta_peak'] = best_theta
        result['itf'] = best_theta[0]
        result['has_theta'] = True
    
    # Find alpha peaks
    alpha_mask = (peaks[:, 0] >= alpha_range[0]) & (peaks[:, 0] <= alpha_range[1])
    if np.any(alpha_mask):
        alpha_peaks = peaks[alpha_mask]
        best_alpha = alpha_peaks[np.argmax(alpha_peaks[:, 1])]
        result['alpha_peak'] = best_alpha
        result['iaf'] = best_alpha[0]
        result['has_alpha'] = True
    
    return result


def get_model_components(model):
    """
    Extract all components from fitted model for plotting.
    
    v5: Fixed to properly extract all model components.
    """
    if model is None:
        return None
    
    try:
        freqs = model.data.freqs
        power_spectrum = model.data.power_spectrum
        modeled_spectrum = model.results.model.modeled_spectrum
        
        # Aperiodic parameters and fit
        ap_params = model.results.params.aperiodic.params
        offset, exponent = ap_params[0], ap_params[1]
        aperiodic_fit = offset - exponent * np.log10(freqs)
        
        # R-squared
        r_squared = model.results.metrics.results.get('gof_rsquared', None)
        
        # Peaks
        peaks = model.results.params.periodic.params
        if peaks is not None and hasattr(peaks, '__len__') and len(peaks) > 0:
            if peaks.ndim == 1:
                peaks = peaks.reshape(1, -1)
        else:
            peaks = np.array([]).reshape(0, 3)
        
        return {
            'freqs': freqs,
            'power_spectrum': power_spectrum,
            'modeled_spectrum': modeled_spectrum,
            'aperiodic_fit': aperiodic_fit,
            'offset': offset,
            'exponent': exponent,
            'r_squared': r_squared,
            'peaks': peaks,
        }
    except Exception as e:
        print(f'Error extracting model components: {e}')
        return None


print('Spectral parameterization ready.')

## 5. Multi-Channel Processing Pipeline

In [ ]:
def process_subject_multichannel(subject_id, eeg_dir=EEG_DIR, verbose=True):
    """
    Process one subject with full multi-channel analysis.
    
    v5: Analyzes all 3 EEG channels (F4, P4, P3) with equal priority.
    """
    sub_info = SUBJECT_INFO[subject_id]
    cb_order = sub_info['counterbalance']
    has_earclip = sub_info['earclip']
    condition_map = RUN_CONDITIONS[cb_order]
    
    run_files = discover_runs(eeg_dir, subject_id)
    
    if len(run_files) == 0:
        return {'subject_id': subject_id, 'error': 'No runs found'}
    
    results = {
        'subject_id': subject_id,
        'counterbalance': cb_order,
        'has_earclip': has_earclip,
        'notes': sub_info.get('notes', ''),
        'n_runs': len(run_files),
        'runs': {},
        'protocol_mismatches': [],
        'specparam_results': [],  # All channel results for diagnostics
    }
    
    # Initialize ITF storage for each channel
    for ch in EEG_CH_LABELS:
        results[f'itf_estimates_{ch}'] = []
        results[f'iaf_estimates_{ch}'] = []
    
    for run_num in sorted(run_files.keys()):
        files = run_files[run_num]
        expected_condition = condition_map.get(run_num, 'unknown')
        
        if verbose:
            print(f'  Run {run_num} ({expected_condition})...', end=' ')
        
        try:
            nic2 = load_nic2_run(files['easy'], files['info'], files['imp'])
        except Exception as e:
            results['runs'][run_num] = {'error': str(e)}
            if verbose:
                print(f'ERROR: {e}')
            continue
        
        # Detect stimulation
        stim_class, stim_info = classify_stimulation(
            nic2['eeg'][:, EEG_CH_IDX[0]], has_earclip, nic2['fs']
        )
        
        # Preprocess
        eeg_filtered, clean_mask, preproc_info = preprocess_eeg(
            nic2['eeg'], has_earclip, nic2['fs']
        )
        
        # Protocol check
        protocol_ok = True
        if expected_condition == 'active' and stim_class != 'ACTIVE':
            protocol_ok = False
            results['protocol_mismatches'].append(run_num)
        
        run_result = {
            'condition': expected_condition,
            'stim_detected': stim_class,
            'protocol_ok': protocol_ok,
            'duration_s': stim_info.get('duration_s', 0),
            'pct_rejected': preproc_info['rejection']['pct_total'],
            'n_clean_samples': preproc_info['n_samples_clean'],
            'early_db': stim_info.get('early_db'),
            'middle_db': stim_info.get('middle_db'),
            'late_db': stim_info.get('late_db'),
        }
        
        # Process baseline and post runs for ITF
        if expected_condition in ['baseline', 'post'] and stim_class != 'ACTIVE':
            eeg_clean = eeg_filtered[clean_mask]
            
            if len(eeg_clean) >= 512:
                # Process ALL channels
                for ch_idx, ch_label in enumerate(EEG_CH_LABELS):
                    freqs, psd = compute_psd(eeg_clean[:, ch_idx])
                    
                    if freqs is not None and psd is not None:
                        model = fit_specparam(freqs, psd)
                        peaks = extract_peaks(model)
                        components = get_model_components(model)
                        
                        r_squared = components['r_squared'] if components else None
                        
                        # Store in run results
                        run_result[f'{ch_label}_itf'] = peaks['itf']
                        run_result[f'{ch_label}_iaf'] = peaks['iaf']
                        run_result[f'{ch_label}_has_theta'] = peaks['has_theta']
                        run_result[f'{ch_label}_has_alpha'] = peaks['has_alpha']
                        run_result[f'{ch_label}_r2'] = r_squared
                        run_result[f'{ch_label}_n_peaks'] = len(peaks['all_peaks']) if peaks['all_peaks'] is not None else 0
                        
                        # Store for aggregation
                        if peaks['has_theta'] and r_squared and r_squared >= ITF_PARAMS['r_squared_threshold']:
                            results[f'itf_estimates_{ch_label}'].append(peaks['itf'])
                        
                        if peaks['has_alpha'] and r_squared and r_squared >= ITF_PARAMS['r_squared_threshold']:
                            results[f'iaf_estimates_{ch_label}'].append(peaks['iaf'])
                        
                        # Store detailed results for diagnostics
                        results['specparam_results'].append({
                            'run': run_num,
                            'condition': expected_condition,
                            'channel': ch_label,
                            'r_squared': r_squared,
                            'itf': peaks['itf'],
                            'iaf': peaks['iaf'],
                            'has_theta': peaks['has_theta'],
                            'has_alpha': peaks['has_alpha'],
                            'all_peaks': peaks['all_peaks'],
                            'model': model,
                            'components': components,
                            'freqs': freqs,
                            'psd': psd,
                        })
        
        results['runs'][run_num] = run_result
        
        if verbose:
            status = '✓' if protocol_ok else '⚠'
            print(f'{status} {stim_class}, {run_result["pct_rejected"]:.1f}% rej')
    
    # Aggregate ITF/IAF across runs for each channel
    for ch_label in EEG_CH_LABELS:
        itf_estimates = results[f'itf_estimates_{ch_label}']
        iaf_estimates = results[f'iaf_estimates_{ch_label}']
        
        if len(itf_estimates) > 0:
            results[f'itf_{ch_label}'] = np.median(itf_estimates)
            results[f'itf_{ch_label}_n'] = len(itf_estimates)
        else:
            results[f'itf_{ch_label}'] = None
            results[f'itf_{ch_label}_n'] = 0
        
        if len(iaf_estimates) > 0:
            results[f'iaf_{ch_label}'] = np.median(iaf_estimates)
            results[f'iaf_{ch_label}_n'] = len(iaf_estimates)
        else:
            results[f'iaf_{ch_label}'] = None
            results[f'iaf_{ch_label}_n'] = 0
    
    # Compute "best" ITF across channels
    all_itf = []
    for ch in EEG_CH_LABELS:
        if results[f'itf_{ch}'] is not None:
            all_itf.append((ch, results[f'itf_{ch}'], results[f'itf_{ch}_n']))
    
    if all_itf:
        # Pick channel with most estimates, or highest ITF if tied
        all_itf.sort(key=lambda x: (-x[2], -x[1]))
        results['best_itf_channel'] = all_itf[0][0]
        results['best_itf'] = all_itf[0][1]
    else:
        results['best_itf_channel'] = None
        results['best_itf'] = None
    
    return results


print('Multi-channel pipeline ready.')

## 6. Single Subject Analysis

In [ ]:
SUBJECT_ID = '10606'

print(f'Processing sub-{SUBJECT_ID}...')
print('=' * 60)

result = process_subject_multichannel(SUBJECT_ID, verbose=True)

print()
print('=' * 60)
print(f'Summary for sub-{SUBJECT_ID}:')
print(f'  Counterbalance: {result["counterbalance"]}')
print(f'  Earclip: {result["has_earclip"]}')
print(f'  Runs processed: {result["n_runs"]}')
print(f'  Protocol mismatches: {result["protocol_mismatches"] if result["protocol_mismatches"] else "None"}')
print()

# Show ITF/IAF for all channels
print('Peak Detection by Channel:')
print(f'{"Channel":>8} {"ITF (Hz)":>12} {"ITF n":>8} {"IAF (Hz)":>12} {"IAF n":>8}')
print('-' * 52)
for ch in EEG_CH_LABELS:
    itf = result.get(f'itf_{ch}')
    itf_n = result.get(f'itf_{ch}_n', 0)
    iaf = result.get(f'iaf_{ch}')
    iaf_n = result.get(f'iaf_{ch}_n', 0)
    
    itf_str = f'{itf:.2f}' if itf else 'None'
    iaf_str = f'{iaf:.2f}' if iaf else 'None'
    
    print(f'{ch:>8} {itf_str:>12} {itf_n:>8} {iaf_str:>12} {iaf_n:>8}')

print()
if result.get('best_itf'):
    print(f'Best ITF: {result["best_itf"]:.2f} Hz (from {result["best_itf_channel"]})')
else:
    print('No theta peaks detected in any channel.')

if result.get('notes'):
    print(f'\n⚠ Note: {result["notes"]}')

In [ ]:
# =============================================================================
# 6b. Multi-Channel Specparam Diagnostic Plot
# =============================================================================

if 'specparam_results' in result and len(result['specparam_results']) > 0:
    # Get unique runs
    runs = sorted(set(r['run'] for r in result['specparam_results']))
    
    # Create subplot grid: rows = runs, cols = channels
    n_runs = len(runs)
    n_channels = len(EEG_CH_LABELS)
    
    fig = make_subplots(
        rows=n_runs, cols=n_channels,
        subplot_titles=[f'Run {r} - {ch}' for r in runs for ch in EEG_CH_LABELS],
        horizontal_spacing=0.05,
        vertical_spacing=0.08,
    )
    
    for row_idx, run_num in enumerate(runs):
        for col_idx, ch_label in enumerate(EEG_CH_LABELS):
            # Find the result for this run/channel
            matches = [r for r in result['specparam_results'] 
                      if r['run'] == run_num and r['channel'] == ch_label]
            
            if not matches:
                continue
            
            res = matches[0]
            components = res['components']
            
            if components is None:
                continue
            
            freqs = components['freqs']
            power = components['power_spectrum']
            ap_fit = components['aperiodic_fit']
            model_fit = components['modeled_spectrum']
            r2 = components['r_squared']
            peaks = components['peaks']
            
            # Plot data
            fig.add_trace(go.Scatter(
                x=freqs, y=power,
                mode='lines', line=dict(color='black', width=1),
                name='Data', showlegend=(row_idx==0 and col_idx==0),
            ), row=row_idx+1, col=col_idx+1)
            
            # Plot aperiodic fit
            fig.add_trace(go.Scatter(
                x=freqs, y=ap_fit,
                mode='lines', line=dict(color='red', width=1, dash='dash'),
                name='Aperiodic', showlegend=(row_idx==0 and col_idx==0),
            ), row=row_idx+1, col=col_idx+1)
            
            # Plot full model
            fig.add_trace(go.Scatter(
                x=freqs, y=model_fit,
                mode='lines', line=dict(color='blue', width=1.5),
                name='Model', showlegend=(row_idx==0 and col_idx==0),
            ), row=row_idx+1, col=col_idx+1)
            
            # Mark peaks
            if len(peaks) > 0:
                for peak in peaks:
                    cf = peak[0]
                    in_theta = 4 <= cf <= 8
                    in_alpha = 8 <= cf <= 13
                    
                    if in_theta:
                        color = 'green'
                        dash = 'solid'
                    elif in_alpha:
                        color = 'purple'
                        dash = 'solid'
                    else:
                        color = 'gray'
                        dash = 'dot'
                    
                    fig.add_vline(
                        x=cf, line=dict(color=color, width=1, dash=dash),
                        row=row_idx+1, col=col_idx+1
                    )
            
            # Add theta band shading
            fig.add_vrect(
                x0=4, x1=8, fillcolor='rgba(0,255,0,0.05)', line_width=0,
                row=row_idx+1, col=col_idx+1
            )
            
            # Add R² annotation
            r2_str = f'R²={r2:.2f}' if r2 else 'No fit'
            fig.add_annotation(
                x=35, y=max(power),
                text=r2_str,
                showarrow=False,
                font=dict(size=9),
                row=row_idx+1, col=col_idx+1,
            )
    
    fig.update_xaxes(range=[1, 40])
    fig.update_layout(
        height=200 * n_runs,
        width=300 * n_channels,
        template='plotly_white',
        title=f'sub-{SUBJECT_ID}: Specparam Fits (All Channels)<br>'
              f'<sub>Green shading = theta (4-8 Hz), Green lines = theta peaks, Purple = alpha</sub>',
        showlegend=True,
        legend=dict(x=1.02, y=0.98),
    )
    
    fig.show()
    
    # Save with descriptive name
    save_plot(fig, 'specparam_multichannel', subject_id=SUBJECT_ID, format='html')
else:
    print('No specparam results to plot.')

In [ ]:
# =============================================================================
# 6c. Detailed Peak Summary Table
# =============================================================================

if 'specparam_results' in result and len(result['specparam_results']) > 0:
    print(f'\nDetailed Peak Summary for sub-{SUBJECT_ID}')
    print('=' * 90)
    print(f'{"Run":>4} {"Cond":>8} {"Ch":>4} {"R²":>6} {"Peaks":>30} {"Theta":>8} {"Alpha":>8}')
    print('-' * 90)
    
    for res in sorted(result['specparam_results'], key=lambda x: (x['run'], x['channel'])):
        peaks = res['all_peaks']
        if peaks is not None and len(peaks) > 0:
            peak_str = ', '.join([f'{p[0]:.1f}' for p in peaks])
        else:
            peak_str = 'None'
        
        r2 = res['r_squared']
        r2_str = f'{r2:.3f}' if r2 else 'N/A'
        
        itf_str = f'{res["itf"]:.1f}' if res['itf'] else '-'
        iaf_str = f'{res["iaf"]:.1f}' if res['iaf'] else '-'
        
        print(f'{res["run"]:>4} {res["condition"]:>8} {res["channel"]:>4} '
              f'{r2_str:>6} {peak_str:>30} {itf_str:>8} {iaf_str:>8}')

## 7. Batch Processing — All Subjects

In [ ]:
all_results = []

print('Processing all subjects (multi-channel)...')
print('=' * 70)

for subject_id in SUBJECT_INFO.keys():
    print(f'\nsub-{subject_id}')
    result = process_subject_multichannel(subject_id, verbose=True)
    all_results.append(result)

print('\n' + '=' * 70)
print('Batch processing complete.')

In [ ]:
# =============================================================================
# 7b. Cross-Subject Peak Detection Summary
# =============================================================================

print('\nCross-Subject Theta/Alpha Detection Summary')
print('=' * 100)

# Collect all peaks across all subjects
all_peaks_data = []

for result in all_results:
    if 'specparam_results' not in result:
        continue
    
    for res in result['specparam_results']:
        peaks = res['all_peaks']
        if peaks is not None and len(peaks) > 0:
            for peak in peaks:
                all_peaks_data.append({
                    'subject_id': result['subject_id'],
                    'earclip': result['has_earclip'],
                    'run': res['run'],
                    'channel': res['channel'],
                    'r_squared': res['r_squared'],
                    'peak_freq': peak[0],
                    'peak_power': peak[1],
                    'peak_bw': peak[2],
                    'in_theta': 4 <= peak[0] <= 8,
                    'in_alpha': 8 <= peak[0] <= 13,
                })

peaks_df = pd.DataFrame(all_peaks_data)

if len(peaks_df) > 0:
    print(f'\nTotal peaks detected: {len(peaks_df)}')
    print(f'  In theta range (4-8 Hz): {peaks_df["in_theta"].sum()}')
    print(f'  In alpha range (8-13 Hz): {peaks_df["in_alpha"].sum()}')
    
    # Summary by subject and channel
    print('\n' + '-' * 100)
    print('Theta Peaks by Subject and Channel:')
    print(f'{"Subject":>10} {"Earclip":>8} {"F4":>8} {"P4":>8} {"P3":>8} {"Best ITF":>12}')
    print('-' * 60)
    
    for result in all_results:
        sid = result['subject_id']
        ec = 'Yes' if result['has_earclip'] else 'No'
        
        f4_itf = result.get('itf_F4')
        p4_itf = result.get('itf_P4')
        p3_itf = result.get('itf_P3')
        best = result.get('best_itf')
        
        f4_str = f'{f4_itf:.1f}' if f4_itf else '-'
        p4_str = f'{p4_itf:.1f}' if p4_itf else '-'
        p3_str = f'{p3_itf:.1f}' if p3_itf else '-'
        best_str = f'{best:.1f} ({result.get("best_itf_channel", "-")})' if best else '-'
        
        print(f'{sid:>10} {ec:>8} {f4_str:>8} {p4_str:>8} {p3_str:>8} {best_str:>12}')
    
    # Peak frequency histogram
    print('\n' + '-' * 100)
    print('Peak Frequency Distribution:')
    
    fig = go.Figure()
    
    fig.add_trace(go.Histogram(
        x=peaks_df['peak_freq'],
        xbins=dict(start=1, end=40, size=1),
        marker_color='steelblue',
        name='All peaks',
    ))
    
    # Add band markers
    fig.add_vrect(x0=4, x1=8, fillcolor='rgba(0,255,0,0.1)', line_width=0,
                  annotation_text='Theta', annotation_position='top left')
    fig.add_vrect(x0=8, x1=13, fillcolor='rgba(128,0,128,0.1)', line_width=0,
                  annotation_text='Alpha', annotation_position='top left')
    fig.add_vline(x=6, line=dict(color='red', dash='dash', width=2),
                  annotation_text='tACS (6 Hz)')
    
    fig.update_layout(
        title='Distribution of Detected Peak Frequencies (All Subjects, All Channels)',
        xaxis_title='Frequency (Hz)',
        yaxis_title='Count',
        template='plotly_white',
        height=400,
        width=800,
    )
    
    fig.show()
    save_plot(fig, 'peak_frequency_distribution', format='html')
    
else:
    print('No peaks detected across any subject.')

In [ ]:
# =============================================================================
# 7c. Group Summary Table
# =============================================================================

summary_rows = []

for result in all_results:
    if 'error' in result:
        summary_rows.append({'subject_id': result['subject_id'], 'status': 'ERROR'})
        continue
    
    rej_rates = [r.get('pct_rejected', 0) for r in result['runs'].values() 
                 if isinstance(r, dict) and 'pct_rejected' in r]
    mean_rej = np.mean(rej_rates) if rej_rates else np.nan
    
    summary_rows.append({
        'subject_id': result['subject_id'],
        'counterbalance': result['counterbalance'],
        'earclip': result['has_earclip'],
        'n_runs': result['n_runs'],
        'protocol_ok': len(result['protocol_mismatches']) == 0,
        'mismatches': str(result['protocol_mismatches']) if result['protocol_mismatches'] else '',
        'mean_pct_rejected': mean_rej,
        'itf_F4': result.get('itf_F4'),
        'itf_P4': result.get('itf_P4'),
        'itf_P3': result.get('itf_P3'),
        'best_itf': result.get('best_itf'),
        'best_ch': result.get('best_itf_channel'),
        'iaf_F4': result.get('iaf_F4'),
        'notes': result.get('notes', ''),
    })

summary_df = pd.DataFrame(summary_rows)

print('\nGroup Summary (Multi-Channel)')
print('=' * 130)
display_cols = ['subject_id', 'earclip', 'protocol_ok', 'mean_pct_rejected', 
                'itf_F4', 'itf_P4', 'itf_P3', 'best_itf', 'best_ch', 'iaf_F4']
print(summary_df[display_cols].to_string(index=False, float_format=lambda x: f'{x:.2f}' if pd.notna(x) else '-'))

# Statistics
print('\n' + '-' * 130)
valid_itf = summary_df.dropna(subset=['best_itf'])
valid_iaf = summary_df.dropna(subset=['iaf_F4'])

print(f'\nTheta Detection: {len(valid_itf)} / {len(summary_df)} subjects with detectable ITF')
if len(valid_itf) > 0:
    print(f'  Mean ITF: {valid_itf["best_itf"].mean():.2f} Hz')
    print(f'  Range: {valid_itf["best_itf"].min():.2f} – {valid_itf["best_itf"].max():.2f} Hz')
    print(f'  Channels used: {valid_itf["best_ch"].value_counts().to_dict()}')

print(f'\nAlpha Detection: {len(valid_iaf)} / {len(summary_df)} subjects with detectable IAF (F4)')
if len(valid_iaf) > 0:
    print(f'  Mean IAF: {valid_iaf["iaf_F4"].mean():.2f} Hz')
    print(f'  Range: {valid_iaf["iaf_F4"].min():.2f} – {valid_iaf["iaf_F4"].max():.2f} Hz')

## 8. Export Results

In [ ]:
# Subject-level summary
summary_path = OUTPUT_DIR / 'group_eeg_summary_v5.csv'
summary_df.to_csv(summary_path, index=False)
print(f'Saved: {summary_path}')

# All detected peaks
if len(peaks_df) > 0:
    peaks_path = OUTPUT_DIR / 'all_detected_peaks_v5.csv'
    peaks_df.to_csv(peaks_path, index=False)
    print(f'Saved: {peaks_path}')

# Run-level details
run_rows = []
for result in all_results:
    if 'runs' not in result:
        continue
    for run_num, run_data in result['runs'].items():
        if isinstance(run_data, dict) and 'error' not in run_data:
            row = {
                'subject_id': result['subject_id'],
                'run': run_num,
                'counterbalance': result['counterbalance'],
                'earclip': result['has_earclip'],
            }
            for k, v in run_data.items():
                if not isinstance(v, (dict, list, np.ndarray)):
                    row[k] = v
            run_rows.append(row)

if run_rows:
    run_df = pd.DataFrame(run_rows)
    run_path = OUTPUT_DIR / 'group_eeg_runs_v5.csv'
    run_df.to_csv(run_path, index=False)
    print(f'Saved: {run_path}')

print('\nExport complete.')

## 9. 6 Hz Timecourse Diagnostic

In [ ]:
PLOT_SUBJECT = '11329'

sub_info = SUBJECT_INFO[PLOT_SUBJECT]
run_files = discover_runs(EEG_DIR, PLOT_SUBJECT)
condition_map = RUN_CONDITIONS[sub_info['counterbalance']]

available_runs = sorted(run_files.keys())
n_runs = len(available_runs)

fig = make_subplots(
    rows=n_runs, cols=1, shared_xaxes=True, vertical_spacing=0.02,
    subplot_titles=[f'Run {r} ({condition_map.get(r, "?")})' for r in available_runs],
)

condition_colors = {
    'baseline': '#1565C0',
    'sham':     '#2E7D32',
    'active':   '#E65100',
    'post':     '#7B1FA2',
}

thresholds = STIM_THRESHOLDS['earclip' if sub_info['earclip'] else 'no_earclip']

for i, run_num in enumerate(available_runs):
    files = run_files[run_num]
    nic2 = load_nic2_run(files['easy'], files['info'], files['imp'])
    condition = condition_map.get(run_num, 'unknown')
    
    t_spec, power_db = compute_6hz_timecourse(nic2['eeg'][:, EEG_CH_IDX[0]])
    stim_class, _ = classify_stimulation(nic2['eeg'][:, EEG_CH_IDX[0]], sub_info['earclip'])
    
    color = condition_colors.get(condition, '#999999')
    
    fig.add_trace(go.Scatter(
        x=t_spec, y=power_db,
        mode='lines', line=dict(color=color, width=1.5),
        name=f'Run {run_num} [{stim_class}]',
    ), row=i+1, col=1)
    
    fig.add_hline(y=thresholds['active_middle_min_db'], 
                  line=dict(color='red', width=0.5, dash='dot'),
                  row=i+1, col=1)
    
    fig.update_yaxes(range=[60, 160], title_text='dB', row=i+1, col=1)

fig.update_xaxes(title_text='Time (s)', row=n_runs, col=1)
fig.update_layout(
    height=120 * n_runs, width=900,
    template='plotly_white',
    title=f'sub-{PLOT_SUBJECT}: 6 Hz Power Timecourse (F4)',
    showlegend=True,
    legend=dict(x=1.02, y=0.5),
)

fig.show()
save_plot(fig, '6hz_timecourse', subject_id=PLOT_SUBJECT, format='html')

---
## Summary: v5 Changes

### 1. Multi-Channel ITF Analysis
All three EEG channels (F4, P4, P3) are now analyzed with equal priority. The "best" ITF is selected based on which channel has the most consistent estimates across runs.

### 2. Fixed Specparam Plotting
Model components (aperiodic fit, full model, peaks) now render correctly. The issue was in how we were extracting and plotting the model components.

### 3. IAF Tracking
In addition to theta peaks (ITF), we now also track alpha peaks (IAF). This provides useful reference data — if we're detecting alpha reliably but not theta, the pipeline is working but theta may genuinely be absent.

### 4. Cross-Subject Diagnostics
New summary tables and histogram showing peak detection across all subjects and channels.

### 5. Descriptive Plot Filenames
All saved plots now use the format: `{name}_sub-{subject_id}_{date}.{format}`